In [1]:
# Core imports for training and evaluation
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# Load the ML feature table exported from feature engineering
ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'ml_df.csv').exists() and (ROOT.parent / 'data' / 'ml_df.csv').exists():
    ROOT = ROOT.parent

ml_df = pd.read_csv(ROOT / 'data' / 'ml_df.csv')
if 'date' in ml_df.columns:
    ml_df['date'] = pd.to_datetime(ml_df['date'], errors='coerce')

# Add ranking_diff if ranking scores exist
if "home_ranking_score" in ml_df.columns and "away_ranking_score" in ml_df.columns:
    ml_df["ranking_diff"] = ml_df["home_ranking_score"] - ml_df["away_ranking_score"]

ml_df.head()


,year,date,tournament_id,tournament_name,match_name,stage,home_team,away_team,home_team_code,away_team_code,...,win_rate_diff,goal_diff_diff,form_diff,goals_per_match_diff,conceded_per_match_diff,season_win_rate_diff,season_goal_diff_diff,elo_diff,result_target,ranking_diff
0,1930,1930-07-13,WC-1930,1930 FIFA World Cup,France v Mexico,Group stage,France,Mexico,FRA,MEX,...,NaN,-3.0,NaN,NaN,NaN,NaN,-3.0,0.0,HomeWin,NaN
1,1930,1930-07-13,WC-1930,1930 FIFA World Cup,United States v Belgium,Group stage,United States,Belgium,USA,BEL,...,NaN,-6.0,NaN,NaN,NaN,NaN,-6.0,0.0,HomeWin,NaN
2,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Romania v Peru,Group stage,Romania,Peru,ROU,PER,...,NaN,-5.0,NaN,NaN,NaN,NaN,-5.0,0.0,HomeWin,NaN
3,1930,1930-07-14,WC-1930,1930 FIFA World Cup,Yugoslavia v Brazil,Group stage,Yugoslavia,Brazil,YUG,BRA,...,NaN,-3.0,NaN,NaN,NaN,NaN,-3.0,0.0,HomeWin,NaN
4,1930,1930-07-15,WC-1930,1930 FIFA World Cup,Argentina v France,Group stage,Argentina,France,ARG,FRA,...,NaN,-2.0,NaN,NaN,NaN,NaN,-2.0,-10.0,HomeWin,NaN


# No-Leakage Validation

Purpose: Explicitly exclude post-match fields and confirm the feature list is safe.

In [3]:
# Define allowed features and remove all post-match leakage columns.
LEAKAGE_COLS = {
    "home_goals",
    "away_goals",
    "score",
    "result",
    "home_team_win",
    "away_team_win",
    "draw",
    "extra_time",
    "penalty_shootout",
    "score_penalties",
}

# Base categorical features allowed
CATEGORICAL_COLS = [
    "stage",
    "tournament_name",
    "host_country",
    "home_team",
    "away_team",
    "tournament_size_category",
]

# Numeric engineered features (derived in long_hist with home_/away_ prefixes)
ENGINEERED_PREFIXES = [
    "home_",
    "away_",
]

feature_cols = [
    c
    for c in ml_df.columns
    if (
        c in CATEGORICAL_COLS
        or any(c.startswith(p) for p in ENGINEERED_PREFIXES)
        or c in [
            "win_rate_diff",
            "goal_diff_diff",
            "form_diff",
            "goals_per_match_diff",
            "conceded_per_match_diff",
            "season_win_rate_diff",
            "season_goal_diff_diff",
            "elo_diff",
            "ranking_diff",
            "total_teams",
            "matches_played",
            "goals_scored_tournament",
            "avg_goals_per_game",
            "year_normalized",
        ]
    )
]

# Remove leakage columns and any accidental target
feature_cols = [c for c in feature_cols if c not in LEAKAGE_COLS and c != "result_target"]

leakage_in_features = set(feature_cols) & LEAKAGE_COLS
print("Leakage columns in features:", leakage_in_features)
print("Feature count:", len(feature_cols))

Leakage columns in features: set()
Feature count: 152


# Time-Aware Train/Test Split

Purpose: Split by tournament year to avoid leakage and simulate real forecasting.

In [4]:
# Create chronological splits for train/validation/test and define X/y.
ml_df = ml_df.sort_values("date").reset_index(drop=True)

train_df = ml_df[ml_df["year"] <= 2014]
val_df = ml_df[ml_df["year"] == 2018]
test_df = ml_df[ml_df["year"] == 2022]

print("Train:", train_df.shape, train_df["year"].min(), "-", train_df["year"].max())
print("Val:", val_df.shape, val_df["year"].min(), "-", val_df["year"].max())
print("Test:", test_df.shape, test_df["year"].min(), "-", test_df["year"].max())

X_train = train_df[feature_cols]
y_train = train_df["result_target"]

X_val = val_df[feature_cols]
y_val = val_df["result_target"]

X_test = test_df[feature_cols]
y_test = test_df["result_target"]

# Build rolling time-based validation folds to reduce overfitting to a single year.
all_years = sorted(ml_df["year"].unique())
cv_val_years = [y for y in all_years if y < 2022][-2:]

folds = []
for vy in cv_val_years:
    fold_train = ml_df[ml_df["year"] < vy]
    fold_val = ml_df[ml_df["year"] == vy]
    folds.append((fold_train, fold_val, vy))

print("CV folds (val years):", cv_val_years)
for fold_train, fold_val, vy in folds:
    print("Fold", vy, "train years", fold_train["year"].min(), "-", fold_train["year"].max(), "val years", fold_val["year"].min(), "-", fold_val["year"].max())

Train: (736, 177) 1930 - 2014
Val: (62, 177) 2018 - 2018
Test: (64, 177) 2022 - 2022
CV folds (val years): [2014, 2018]
Fold 2014 train years 1930 - 2010 val years 2014 - 2014
Fold 2018 train years 1930 - 2014 val years 2018 - 2018


# Model Training

Purpose: Fit multiple candidate models and compare on validation log loss.

In [5]:
# Build preprocessing pipelines and train baseline/ensemble models.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    brier_score_loss,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    num_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        transformers=[("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)]
    )


from sklearn.base import clone

preprocessor = build_preprocessor(X_train)

models = {
    "LogReg": LogisticRegression(
        solver="saga",
        max_iter=5000,
        C=0.5,
        multi_class="multinomial",
        class_weight="balanced",
        n_jobs=-1,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=600,
        max_depth=8,
        min_samples_leaf=5,
        max_features=0.3,
        class_weight="balanced_subsample",
        random_state=42,
    ),
}

if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        n_estimators=800,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=5,
        subsample=0.7,
        colsample_bytree=0.7,
        gamma=0.5,
        reg_lambda=2.0,
        reg_alpha=0.1,
        random_state=42,
    )

if LGBM_AVAILABLE:
    models["LightGBM"] = LGBMClassifier(
        objective="multiclass",
        n_estimators=800,
        learning_rate=0.05,
        num_leaves=15,
        max_depth=4,
        min_data_in_leaf=30,
        feature_fraction=0.7,
        bagging_fraction=0.7,
        bagging_freq=1,
        lambda_l1=0.1,
        lambda_l2=2.0,
        random_state=42,
    )
else:
    models["HistGB"] = HistGradientBoostingClassifier(
        max_depth=3,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=0.1,
        random_state=42,
    )


CLASS_ORDER = ["HomeWin", "Draw", "AwayWin"]

label_encoder = LabelEncoder()
label_encoder.fit(CLASS_ORDER)

CLASS_ORDER_ENC = list(label_encoder.transform(CLASS_ORDER))

y_train_enc = label_encoder.transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)


def get_sample_weight(y_series):
    classes = np.array(CLASS_ORDER)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_series)
    weight_map = {cls: w for cls, w in zip(classes, weights)}
    return y_series.map(weight_map).to_numpy()


def get_sample_weight_enc(y_enc):
    classes = np.array(CLASS_ORDER_ENC)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_enc)
    weight_map = {cls: w for cls, w in zip(classes, weights)}
    return np.array([weight_map[v] for v in y_enc])


class PrefitPipeline:
    def __init__(self, preprocessor, model):
        self.preprocessor = preprocessor
        self.model = model

    def fit(self, X, y=None):
        # no-op for prefit estimator to satisfy sklearn interface
        return self

    def predict(self, X):
        Xp = self.preprocessor.transform(X)
        return self.model.predict(Xp)

    def predict_proba(self, X):
        Xp = self.preprocessor.transform(X)
        return self.model.predict_proba(Xp)


def evaluate_model(pipe, X, y_true, class_order) -> dict:
    preds = pipe.predict(X)
    proba = pipe.predict_proba(X)

    acc = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds, average="macro")
    ll = log_loss(y_true, proba, labels=class_order)
    brier = np.mean(
        [brier_score_loss((y_true == c).astype(int), proba[:, i]) for i, c in enumerate(class_order)]
    )
    y_bin = pd.get_dummies(y_true).reindex(columns=class_order, fill_value=0)
    try:
        roc = roc_auc_score(
            y_bin,
            proba,
            multi_class="ovr",
        )
    except Exception:
        roc = np.nan
    cm = confusion_matrix(y_true, preds, labels=class_order)

    return {
        "accuracy": acc,
        "macro_f1": f1,
        "log_loss": ll,
        "brier": brier,
        "roc_auc_ovr": roc,
        "confusion_matrix": cm,
    }


def fit_with_early_stopping(name, model, preprocessor, X_tr, y_tr, X_va, y_va, sample_weight=None):
    X_tr_p = preprocessor.fit_transform(X_tr)
    X_va_p = preprocessor.transform(X_va)

    if name == "XGBoost":
        try:
            model.fit(
                X_tr_p,
                y_tr,
                sample_weight=sample_weight,
                eval_set=[(X_va_p, y_va)],
                verbose=False,
                early_stopping_rounds=50,
            )
        except TypeError:
            # Older xgboost versions don't support early_stopping_rounds in sklearn API
            model.fit(
                X_tr_p,
                y_tr,
                sample_weight=sample_weight,
                eval_set=[(X_va_p, y_va)],
                verbose=False,
            )
    elif name == "LightGBM":
        try:
            model.fit(
                X_tr_p,
                y_tr,
                sample_weight=sample_weight,
                eval_set=[(X_va_p, y_va)],
                eval_metric="multi_logloss",
                verbose=False,
                early_stopping_rounds=50,
            )
        except TypeError:
            # Older lightgbm versions may not support verbose/early_stopping_rounds
            model.fit(
                X_tr_p,
                y_tr,
                sample_weight=sample_weight,
                eval_set=[(X_va_p, y_va)],
                eval_metric="multi_logloss",
            )
    else:
        model.fit(X_tr_p, y_tr, sample_weight=sample_weight)

    pipe = PrefitPipeline(preprocessor, model)
    # Mark as fitted for sklearn calibration compatibility
    if hasattr(model, "classes_"):
        pipe.classes_ = model.classes_
    else:
        pipe.classes_ = np.unique(y_tr)
    return pipe


# Time-series CV evaluation to reduce overfitting to a single year
results = []

for fold_train, fold_val, fold_year in folds:
    # Baseline: always predict HomeWin
    y_va_base = fold_val["result_target"]
    base_pred = pd.Series(["HomeWin"] * len(y_va_base), index=y_va_base.index)
    base_proba = np.zeros((len(y_va_base), len(CLASS_ORDER)))
    base_proba[:, CLASS_ORDER.index("HomeWin")] = 1.0
    base_metrics = {
        "accuracy": accuracy_score(y_va_base, base_pred),
        "macro_f1": f1_score(y_va_base, base_pred, average="macro"),
        "log_loss": log_loss(y_va_base, base_proba, labels=CLASS_ORDER),
        "brier": np.mean(
            [brier_score_loss((y_va_base == c).astype(int), base_proba[:, i]) for i, c in enumerate(CLASS_ORDER)]
        ),
        "roc_auc_ovr": np.nan,
        "confusion_matrix": confusion_matrix(y_va_base, base_pred, labels=CLASS_ORDER),
    }
    results.append({"model": "Baseline_HomeWin", "fold_year": fold_year, **base_metrics})

    for name, model in models.items():
        X_tr = fold_train[feature_cols]
        X_va = fold_val[feature_cols]

        if name == "XGBoost":
            y_tr = label_encoder.transform(fold_train["result_target"])
            y_va = label_encoder.transform(fold_val["result_target"])
            sw_tr = get_sample_weight_enc(y_tr)
            preprocessor_fold = build_preprocessor(X_tr)
            model_fold = clone(model)
            pipe = fit_with_early_stopping(
                name,
                model_fold,
                preprocessor_fold,
                X_tr,
                y_tr,
                X_va,
                y_va,
                sample_weight=sw_tr,
            )
            metrics = evaluate_model(pipe, X_va, y_va, CLASS_ORDER_ENC)
        else:
            y_tr = fold_train["result_target"]
            y_va = fold_val["result_target"]
            sw_tr = get_sample_weight(y_tr)
            preprocessor_fold = build_preprocessor(X_tr)
            model_fold = clone(model)

            if name in ["LightGBM", "HistGB"]:
                pipe = fit_with_early_stopping(
                    name,
                    model_fold,
                    preprocessor_fold,
                    X_tr,
                    y_tr,
                    X_va,
                    y_va,
                    sample_weight=sw_tr,
                )
            else:
                pipe = Pipeline(steps=[("prep", preprocessor_fold), ("model", model_fold)])
                pipe.fit(X_tr, y_tr, model__sample_weight=sw_tr)

            metrics = evaluate_model(pipe, X_va, y_va, CLASS_ORDER)

        results.append({"model": name, "fold_year": fold_year, **metrics})

results_df = pd.DataFrame(results)
summary_df = (
    results_df.groupby("model")[["accuracy", "macro_f1", "log_loss", "brier", "roc_auc_ovr"]]
    .mean()
    .sort_values("log_loss")
)
summary_df

/Users/jeanphilippeauguste/Downloads/Techlabs Dusseldorf Data Science/work/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightG

/Users/jeanphilippeauguste/Downloads/Techlabs Dusseldorf Data Science/work/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/jeanphilippeauguste/Downloads/Techlabs Dusseldorf Data Science/work/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/jeanphilippeauguste/Downloads/Techlabs Dusseldorf Data Science/work/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] min_data_in_leaf is set=30, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=30
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] lambda_l2 is set=2.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.0
[LightG

/Users/jeanphilippeauguste/Downloads/Techlabs Dusseldorf Data Science/work/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/jeanphilippeauguste/Downloads/Techlabs Dusseldorf Data Science/work/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,accuracy,macro_f1,log_loss,brier,roc_auc_ovr
model,,,,,
XGBoost,0.879672,0.859356,0.273482,0.475241,0.416186
LightGBM,0.903738,0.887223,0.274505,0.513670,0.405904
RandomForest,0.879800,0.862431,0.390013,0.419787,0.388994
LogReg,0.344086,0.251137,1.392266,0.300155,0.380988
Baseline_HomeWin,0.423835,0.198350,22.483451,0.384110,NaN


# Model Evaluation

Purpose: Select the best model on validation log loss and report test metrics.

In [6]:
# Display CV-averaged ranking table (lower log_loss is better).
summary_df

,accuracy,macro_f1,log_loss,brier,roc_auc_ovr
model,,,,,
XGBoost,0.879672,0.859356,0.273482,0.475241,0.416186
LightGBM,0.903738,0.887223,0.274505,0.513670,0.405904
RandomForest,0.879800,0.862431,0.390013,0.419787,0.388994
LogReg,0.344086,0.251137,1.392266,0.300155,0.380988
Baseline_HomeWin,0.423835,0.198350,22.483451,0.384110,NaN


In [7]:
# Pick the best model (by CV log loss) and evaluate on the 2022 test set.
cv_ranked = summary_df.drop(index=["Baseline_HomeWin"], errors="ignore")
best_model_name = cv_ranked.index[0]
print("Best model by CV log loss:", best_model_name)

# Fit the selected model on the original train split (<=2014), using 2018 as early-stopping/validation.
if best_model_name == "XGBoost":
    final_preprocessor = build_preprocessor(X_train)
    final_model = clone(models[best_model_name])
    sw_tr = get_sample_weight_enc(y_train_enc)
    best_model = fit_with_early_stopping(
        best_model_name,
        final_model,
        final_preprocessor,
        X_train,
        y_train_enc,
        X_val,
        y_val_enc,
        sample_weight=sw_tr,
    )
    val_metrics = evaluate_model(best_model, X_val, y_val_enc, CLASS_ORDER_ENC)
    test_metrics = evaluate_model(best_model, X_test, y_test_enc, CLASS_ORDER_ENC)
else:
    final_preprocessor = build_preprocessor(X_train)
    final_model = clone(models[best_model_name])
    sw_tr = get_sample_weight(y_train)

    if best_model_name in ["LightGBM", "HistGB"]:
        best_model = fit_with_early_stopping(
            best_model_name,
            final_model,
            final_preprocessor,
            X_train,
            y_train,
            X_val,
            y_val,
            sample_weight=sw_tr,
        )
    else:
        best_model = Pipeline(steps=[("prep", final_preprocessor), ("model", final_model)])
        best_model.fit(X_train, y_train, model__sample_weight=sw_tr)

    val_metrics = evaluate_model(best_model, X_val, y_val, CLASS_ORDER)
    test_metrics = evaluate_model(best_model, X_test, y_test, CLASS_ORDER)

test_metrics

Best model by CV log loss: XGBoost


{'accuracy': 0.890625,
 'macro_f1': 0.8817766288261962,
 'log_loss': 0.27575135584258065,
 'brier': 0.4453644754541141,
 'roc_auc_ovr': 0.4328256171852231,
 'confusion_matrix': array([[26,  2,  1],
        [ 0, 13,  2],
        [ 1,  1, 18]])}

In [8]:
# Calibrate probabilities on validation data and plot reliability curves.
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Probability calibration using validation split
base_proba = best_model.predict_proba(X_val)

if best_model_name == "XGBoost":
    base_log_loss = log_loss(y_val_enc, base_proba, labels=CLASS_ORDER_ENC)
    base_brier = np.mean(
        [brier_score_loss((y_val_enc == c).astype(int), base_proba[:, i]) for i, c in enumerate(CLASS_ORDER_ENC)]
    )

    calibrated = CalibratedClassifierCV(best_model, method="sigmoid", cv="prefit")
    calibrated.fit(X_val, y_val_enc)

    cal_proba = calibrated.predict_proba(X_val)
    cal_log_loss = log_loss(y_val_enc, cal_proba, labels=CLASS_ORDER_ENC)
    cal_brier = np.mean(
        [brier_score_loss((y_val_enc == c).astype(int), cal_proba[:, i]) for i, c in enumerate(CLASS_ORDER_ENC)]
    )
else:
    base_log_loss = log_loss(y_val, base_proba, labels=CLASS_ORDER)
    base_brier = np.mean(
        [brier_score_loss((y_val == c).astype(int), base_proba[:, i]) for i, c in enumerate(CLASS_ORDER)]
    )

    calibrated = CalibratedClassifierCV(best_model, method="sigmoid", cv="prefit")
    calibrated.fit(X_val, y_val)

    cal_proba = calibrated.predict_proba(X_val)
    cal_log_loss = log_loss(y_val, cal_proba, labels=CLASS_ORDER)
    cal_brier = np.mean(
        [brier_score_loss((y_val == c).astype(int), cal_proba[:, i]) for i, c in enumerate(CLASS_ORDER)]
    )

print("Log loss before:", round(base_log_loss, 4), "after:", round(cal_log_loss, 4))
print("Brier before:", round(base_brier, 4), "after:", round(cal_brier, 4))

# Calibration curve (macro average)
plt.figure(figsize=(6, 6))
for i, c in enumerate(CLASS_ORDER):
    frac_pos, mean_pred = calibration_curve((y_val == c).astype(int), cal_proba[:, i], n_bins=10)
    plt.plot(mean_pred, frac_pos, marker="o", label=c)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("Calibration Curve (Validation)")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.legend()
plt.show()


/Users/jeanphilippeauguste/Downloads/Techlabs Dusseldorf Data Science/work/.venv/lib/python3.9/site-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


ValueError: PrefitPipeline should either be a classifier to be used with response_method=['decision_function', 'predict_proba'] or the response_method should be 'predict'. Got a regressor with response_method=['decision_function', 'predict_proba'] instead.

# Score Prediction (Regression)

Purpose: Train models to predict home and away goals so we can output a scoreline.

In [9]:
# Train regressors to predict home/away goals (pre-match features only)
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import FunctionTransformer

# Targets
y_home = train_df["home_goals"]
y_away = train_df["away_goals"]

# Densify because HistGradientBoostingRegressor doesn't accept sparse matrices
def to_dense_fn(x):
    return x.toarray() if hasattr(x, "toarray") else x


to_dense = FunctionTransformer(to_dense_fn, accept_sparse=True)

reg_home = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("to_dense", to_dense),
        (
            "model",
            HistGradientBoostingRegressor(
                max_depth=5,
                learning_rate=0.05,
                max_leaf_nodes=31,
                min_samples_leaf=20,
                l2_regularization=0.0,
                random_state=42,
            ),
        ),
    ]
)
reg_away = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("to_dense", to_dense),
        (
            "model",
            HistGradientBoostingRegressor(
                max_depth=5,
                learning_rate=0.05,
                max_leaf_nodes=31,
                min_samples_leaf=20,
                l2_regularization=0.0,
                random_state=42,
            ),
        ),
    ]
)

reg_home.fit(X_train, y_home)
reg_away.fit(X_train, y_away)

# Evaluate on 2022 test set
pred_home = reg_home.predict(X_test)
pred_away = reg_away.predict(X_test)

mae_home = mean_absolute_error(test_df["home_goals"], pred_home)
rmse_home = np.sqrt(mean_squared_error(test_df["home_goals"], pred_home))

mae_away = mean_absolute_error(test_df["away_goals"], pred_away)
rmse_away = np.sqrt(mean_squared_error(test_df["away_goals"], pred_away))

score_metrics = pd.DataFrame({
    "target": ["home_goals", "away_goals"],
    "mae": [mae_home, mae_away],
    "rmse": [rmse_home, rmse_away],
})

score_metrics

,target,mae,rmse
0,home_goals,0.597273,0.789097
1,away_goals,0.523396,0.701882


In [10]:
# Persist the final model artifact for downstream inference.
import joblib

MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Use calibrated model only if calibration metrics are available
if "cal_log_loss" in globals() and "base_log_loss" in globals():
    final_model = calibrated if cal_log_loss <= base_log_loss else best_model
else:
    final_model = best_model

artifact = {
    "model": final_model,
    "feature_cols": feature_cols,
    "class_order": CLASS_ORDER,
    "score_models": {
        "home_goals": reg_home,
        "away_goals": reg_away,
    },
}

artifact_path = MODELS_DIR / "worldcup_model2.joblib"
joblib.dump(artifact, artifact_path)

print("Saved model artifact to:", artifact_path)


NameError: name 'cal_log_loss' is not defined

In [11]:
# save with pickle
import pickle

pickle_path = MODELS_DIR / "worldcup_model2.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(artifact, f)

print("Saved pickle artifact to:", pickle_path)

NameError: name 'artifact' is not defined